# 06 — Final analysis (post-session, report material)

Runs **after** the single §0.7 test session, on the per-window CSVs it
wrote. **No test access of its own**: it reads files, never a checkpoint
and never the dataset, so it adds no line to `test_invocations.jsonl` and
can be re-run freely.

Covers the two zero-cost consolidation directions (`CONSOLIDATION_REVIEW.md`):

- **D — model characterization:** per-class and per-AR-set error structure,
  calibration (reliability + ECE). macro-F1 averages 8 classes equally and
  hides whether the model is uniformly mediocre or strong on 6 and blind on
  2; this is also where the declared "val is blind to C/E/S" caveat gets
  tested empirically — do those classes actually underperform on test?
- **A — paired comparison:** trace-level (cluster) paired bootstrap on the
  difference between two streams.

**Row-count agnostic:** it analyses whatever streams have CSVs, so a 13- or
14-row session both work without edits.

## Two methodological constraints, decided before seeing any number

**1. The bootstrap runs on ACCURACY, not macro-F1.** `harness.macro_f1`
averages only over classes present in the ground truth. On the primary test
(11 traces, 6 of 8 classes with a single trace each) a trace-level resample
contains all 8 classes only ~2.8% of the time (measured, 20k resamples), so
a macro-F1 bootstrap would average a *different class set* in ~97% of
replicates — the estimand changes replicate to replicate and the interval
means nothing. Accuracy is well defined whatever classes appear. macro-F1 is
still reported, descriptively, without an interval.

**2. This measures ONE of the two variances.** The bootstrap resamples the
*test set* with the trained models held fixed: it answers "how much does the
gap move if we had drawn different test traces?". It says nothing about
*training/seed* variance — the 5.45-pt C2 vs C2_s43 swing (E1′) lives there.
Report them as two separate, labelled uncertainties; conflating them would
overstate what either shows.

## Environment setup

No staging, no GPU, no checkpoints: this notebook only reads CSVs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd

SEED = 42
N_BOOT = 10000  # fixed a priori
print("Repo:", REPO_DIR)

## Analysis functions

Notebook-local by the project's dividing line (`notebooks/diagnostics/README.md`):
new math, but run once for the report rather than re-run across sessions. If
the team decides these numbers deserve package status, the move is
mechanical — same path NCM/kNN took into `sharp_har/diagnostics.py`.

In [ ]:
def per_class_report(df, labels):
    """Per-class precision/recall/F1 + support from a harness windows CSV.
    Classes absent from y_true are reported with support 0 (the declared
    val-blind classes are exactly these)."""
    rows = []
    for i, lab in enumerate(labels):
        tp = int(((df.y_pred == i) & (df.y_true == i)).sum())
        fp = int(((df.y_pred == i) & (df.y_true != i)).sum())
        fn = int(((df.y_pred != i) & (df.y_true == i)).sum())
        prec = tp / (tp + fp) if tp + fp else float("nan")
        rec = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float("nan")
        rows.append({"class": lab, "support": int((df.y_true == i).sum()),
                     "precision": prec, "recall": rec, "f1": f1})
    return pd.DataFrame(rows)


def calibration(df, labels, n_bins=10):
    """Reliability table + ECE from the fused per-class probabilities the
    harness stores as p_<label>. Confidence = max fused probability;
    ECE = sum_b (n_b/N) * |acc_b - conf_b| (equal-width bins)."""
    P = df[[f"p_{l}" for l in labels]].to_numpy(dtype=float)
    conf = P.max(axis=1)
    correct = (P.argmax(axis=1) == df.y_true.to_numpy()).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if not m.any():
            continue
        acc_b, conf_b, n_b = correct[m].mean(), conf[m].mean(), int(m.sum())
        ece += (n_b / len(df)) * abs(acc_b - conf_b)
        rows.append({"bin": f"({lo:.1f},{hi:.1f}]", "n": n_b,
                     "confidence": conf_b, "accuracy": acc_b, "gap": acc_b - conf_b})
    return pd.DataFrame(rows), ece


def paired_bootstrap(df_a, df_b, n_boot=N_BOOT, seed=SEED):
    """Trace-level (cluster) PAIRED bootstrap of the accuracy difference
    A - B. Resamples TRACES with replacement (never windows: windows within
    a trace are correlated, and window-level resampling would contradict the
    split-by-trace principle and understate variance), and applies the SAME
    resample to both streams so per-trace difficulty cancels.

    Returns the observed difference, the percentile CI, and the fraction of
    replicates favouring A. Accuracy, not macro-F1 — see the header."""
    a = df_a.set_index(["trace_id", "window_start"]).sort_index()
    b = df_b.set_index(["trace_id", "window_start"]).sort_index()
    assert a.index.equals(b.index), (
        "the two streams do not cover the same fused windows — different split "
        "or set; a paired comparison is only defined on identical units"
    )
    ok_a = (a.y_pred == a.y_true).to_numpy()
    ok_b = (b.y_pred == b.y_true).to_numpy()
    traces = a.index.get_level_values(0).to_numpy()
    uniq = np.unique(traces)
    idx_of = {t: np.flatnonzero(traces == t) for t in uniq}

    observed = ok_a.mean() - ok_b.mean()
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for k in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        sel = np.concatenate([idx_of[t] for t in pick])
        diffs[k] = ok_a[sel].mean() - ok_b[sel].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"observed": observed, "ci_low": lo, "ci_high": hi,
            "frac_favouring_a": float((diffs > 0).mean()), "n_traces": len(uniq)}

## Self-test on synthetic data

Run this **before** the session: it proves the three functions behave on
inputs whose answers are known, so on session day the code is already
trusted. Same discipline as the day-2/day-3 synthetic verification.

In [ ]:
def _synth(n_traces=11, per_trace=40, n_cls=8, acc=0.8, seed=0):
    """Fake windows CSV: one activity per trace (as in the real test set).
    Confidence is pinned at 1 - 0.05*(n_cls-1) = 0.65 regardless of `acc`,
    which is what makes the calibration check below meaningful."""
    rng = np.random.default_rng(seed)
    labels = [chr(65 + i) for i in range(n_cls)]
    rows = []
    for t in range(n_traces):
        y = t % n_cls
        for w in range(per_trace):
            right = rng.random() < acc
            pred = y if right else int(rng.choice([c for c in range(n_cls) if c != y]))
            p = np.full(n_cls, 0.05); p[pred] = 1.0 - 0.05 * (n_cls - 1)
            rows.append({"trace_id": f"T{t}", "window_start": w * 340, "ar_set": "AR-7",
                         "y_true": y, "y_pred": pred,
                         **{f"p_{labels[c]}": p[c] for c in range(n_cls)}})
    return pd.DataFrame(rows), labels

df_hi, labels_s = _synth(acc=0.90, seed=1)
df_lo, _ = _synth(acc=0.60, seed=1)

# 1. per-class: support must sum to N, and a perfect stream must give F1 = 1
pc = per_class_report(df_hi, labels_s)
assert pc["support"].sum() == len(df_hi)
df_perf = df_hi.copy(); df_perf["y_pred"] = df_perf["y_true"]
assert np.allclose(per_class_report(df_perf, labels_s)["f1"].dropna(), 1.0)

# 2. calibration: ECE must track the |accuracy - confidence| GAP, not the
#    accuracy. Both synthetic streams sit at confidence 0.65, so the
#    90%-accurate one is the MORE miscalibrated (underconfident by 0.25)
#    and the 60%-accurate one the better calibrated (gap 0.05). A naive
#    implementation that conflates ECE with error rate fails this.
_, ece_hi = calibration(df_hi, labels_s)
_, ece_lo = calibration(df_lo, labels_s)
assert ece_hi > ece_lo, (ece_hi, ece_lo)
assert abs(ece_hi - 0.25) < 0.02 and abs(ece_lo - 0.05) < 0.02, (ece_hi, ece_lo)

# 3. paired bootstrap: A clearly better than B -> CI entirely above 0;
#    a stream against ITSELF -> observed 0 and a degenerate CI at 0
r_ab = paired_bootstrap(df_hi, df_lo)
r_aa = paired_bootstrap(df_hi, df_hi)
assert r_ab["observed"] > 0 and r_ab["ci_low"] > 0, r_ab
assert abs(r_aa["observed"]) < 1e-12 and r_aa["ci_low"] == r_aa["ci_high"] == 0.0, r_aa

print("self-test passed")
print(f"  A(0.90) vs B(0.60): diff {r_ab['observed']:+.3f} "
      f"CI [{r_ab['ci_low']:+.3f}, {r_ab['ci_high']:+.3f}] over {r_ab['n_traces']} traces")
print(f"  ECE: 90%-accurate {ece_hi:.3f} (underconfident) | "
      f"60%-accurate {ece_lo:.3f} (well matched to conf 0.65)")

## Load the session's CSVs

Point `FINAL_DIR` at `reports/final/` (after the session commit) or at the
Drive run folders. Every `*_windows.csv` found becomes a stream; the stream
name is taken from the filename, so a 14-row session needs no edit here.

In [ ]:
FINAL_DIR = REPO_DIR / "reports" / "final"
assert FINAL_DIR.exists(), (
    f"{FINAL_DIR} not found — run this AFTER the §0.7 session has been committed "
    "(or point FINAL_DIR at the Drive run folders)."
)

streams = {}
for p in sorted(FINAL_DIR.glob("*_windows.csv")):
    if "_test_" not in p.name:
        continue  # val CSVs, if any, stay out of the report analysis
    name = p.name.split("_test_")[0]
    streams[name] = pd.read_csv(p)

assert streams, f"no *_test_*_windows.csv under {FINAL_DIR}"
LABELS = [c[2:] for c in next(iter(streams.values())).columns if c.startswith("p_")]
print(f"{len(streams)} streams | classes {LABELS}")
for k, v in streams.items():
    acc = (v.y_pred == v.y_true).mean()
    print(f"  {k:24s} {len(v):5d} fused windows, {v.trace_id.nunique():3d} traces, acc {acc:.4f}")

## D — error structure and calibration

In [ ]:
FOCUS = "C1"  # the deliverable model; change to inspect another stream
df = streams[FOCUS]

print(f"=== {FOCUS}: per-class ===")
print(per_class_report(df, LABELS).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\nDeclared caveat to check here: val was blind to C/E/S — do they underperform?")

print(f"\n=== {FOCUS}: per-AR-set ===")
for s, g in df.groupby("ar_set"):
    print(f"  {s}: {len(g):5d} windows, acc {(g.y_pred == g.y_true).mean():.4f}")
print("(degenerate on the primary test — a single AR-set; informative on C0 and S6-out)")

rel, ece = calibration(df, LABELS)
print(f"\n=== {FOCUS}: calibration — ECE {ece:.4f} ===")
print(rel.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("gap > 0 = underconfident, gap < 0 = overconfident in that bin")

## A — paired bootstrap between streams

Only pairs on the **same split and set** are comparable (C0 is P1, S6-out is
P2-living — never pair those with the P2-lab streams). The assert inside
`paired_bootstrap` refuses mismatched units rather than silently aligning.

In [ ]:
PAIRS = [("C1", "C2"), ("C1", "C3"), ("C1", "C3_ft"), ("C1", "C1_AdaBN"), ("C1", "C1_T3A")]

print(f"{'comparison':22s} {'diff':>8s} {'95% CI':>20s} {'P(A>B)':>8s}")
print("-" * 62)
for a, b in PAIRS:
    if a not in streams or b not in streams:
        print(f"{a} vs {b:14s}   (missing — skipped)")
        continue
    r = paired_bootstrap(streams[a], streams[b])
    print(f"{a+' vs '+b:22s} {r['observed']:+8.4f} "
          f"[{r['ci_low']:+.4f}, {r['ci_high']:+.4f}] {r['frac_favouring_a']:8.3f}")
print("-" * 62)
print("CI crossing 0 = the accuracy gap is not resolved by this test sample.")
print("This is TEST-SAMPLING variance only; seed variance (E1': 5.45 pts on C2)")
print("is a separate, unmeasured-here uncertainty. Report both, never merge them.")

## Archiving

Executed copy → `notebooks/runs/YYYY-MM-DD_final_analysis.ipynb` + the
numbers into the report. No test access happened here, so there is nothing
to log in `test_invocations.jsonl` — and re-running is free.